# 2. SmolVLM-256M-Instruct: Baselines & Fine-Tuning

This notebook covers the complete experimental pipeline for **SmolVLM-256M-Instruct**:
1. **Zero-shot and one-shot inference:** baselines before fine-tuning.
2. **Supervised Fine-Tuning (SFT) with LoRA:** two configurations (1 dataset / 2 datasets).
3. **Checkpoint selection** on the validation set.
4. **Final evaluation** on the test set (70 examples).

All results are saved to Google Drive for the comparison notebook (`4_comparison.ipynb`).

SmolVLM-256M-Instruct (SmolVLM Team, 2025a) is a compact 256M-parameter vision-language model designed for efficient multimodal tasks. Its small size makes it suitable for fine-tuning on consumer-grade GPUs.

**Shared modules:** This notebook imports `data_prep.py` (dataset loading) and `metrics.py` (evaluation metrics). Prompt templates and the evaluation pipeline are model-specific and defined in this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip uninstall -y huggingface-hub transformers gradio torchao
!pip install -q huggingface-hub>=1.2.0 torchao>=0.16.0 datasets pillow sacrebleu \
    accelerate bitsandbytes peft triton transformers==4.46.0 editdistance

Found existing installation: huggingface_hub 1.27.0
Uninstalling huggingface_hub-1.27.0:
  Successfully uninstalled huggingface_hub-1.27.0
Found existing installation: transformers 5.15.0
Uninstalling transformers-5.15.0:
  Successfully uninstalled transformers-5.15.0
Found existing installation: gradio 6.24.0
Uninstalling gradio-6.24.0:
  Successfully uninstalled gradio-6.24.0
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet


In [ ]:
import gc
import json
import os
import sys
import time
from collections import Counter
from glob import glob

import datasets
import editdistance
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sacrebleu
import torch
from datasets import Dataset, concatenate_datasets
from peft import LoraConfig, PeftModel, get_peft_model
from PIL import Image
from tqdm import tqdm
from transformers import (
    AutoModelForVision2Seq,
    AutoProcessor,
    Trainer,
    TrainingArguments,
)

# Ensure shared modules are importable
if os.path.exists("metrics.py"):
    sys.path.insert(0, ".")
else:
    sys.path.insert(0, "/content/drive/MyDrive/latex_ocr_project")
import data_prep
from metrics import compute_metrics, normalize_latex

In [ ]:
SEED = 42
MW_SUBSAMPLE_SIZE = 5000
MODEL_NAME = "HuggingFaceTB/SmolVLM-256M-Instruct"
SAVE_DIR = "/content/drive/MyDrive/latex_ocr"

## 1. Data Loading

We use the `data_prep` module to load the LaTeX_OCR dataset (train/val/test splits) and a 5 K-subsample of the MathWriting-human dataset. Both datasets are described and analyzed in `1_eda_and_setup.ipynb` (Sections 1.1 and 1.2). The primary LaTeX_OCR dataset provides 70 test examples for evaluation. The MathWriting-human dataset supplements training data with 5,000 handwritten expressions.

In [ ]:
dataset = data_prep.load_latex_ocr()
mw_train_subsample = data_prep.load_mathwriting_subsample(
    subsample_size=MW_SUBSAMPLE_SIZE,
    random_state=SEED,
)

print("LaTeX_OCR splits:")
for split in dataset:
    print(f"  {split}: {len(dataset[split]):,} examples")
print(f"\nMathWriting subsample: {len(mw_train_subsample):,} examples")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

human_handwrite/train-00000-of-00001.par(…):   0%|          | 0.00/16.2M [00:00<?, ?B/s]

human_handwrite/validation-00000-of-0000(…):   0%|          | 0.00/961k [00:00<?, ?B/s]

human_handwrite/test-00000-of-00001.parq(…):   0%|          | 0.00/906k [00:00<?, ?B/s]

...

data/val-00000-of-00001-184984e66f80ed7a(…):   0%|          | 0.00/81.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/229864 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7644 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/15674 [00:00<?, ? examples/s]

MathWriting subsample: 5,000 examples, columns: ['image', 'text']
LaTeX_OCR splits:
  train: 1,164 examples
  validation: 68 examples
  test: 70 examples

MathWriting subsample: 5,000 examples


## 2. Zero- and One-shot Inference

### 2.1. Prompt Template and Helper Functions

In [ ]:
def build_zero_shot_prompt(processor):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": "Convert this handwritten mathematical formula to LaTeX format. Output only the LaTeX code, nothing else."}
            ]
        }
    ]
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [ ]:
def build_one_shot_prompt(processor, example_image, example_latex):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": "Convert this handwritten mathematical formula to LaTeX format. Output only the LaTeX code, nothing else."}
            ]
        },
        {
            "role": "assistant",
            "content": example_latex
        },
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": "Convert this handwritten mathematical formula to LaTeX format. Output only the LaTeX code, nothing else."}
            ]
        }
    ]
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [ ]:
def predict(images, prompt_text, model, processor, device="cpu", max_new_tokens=256):
    """
    Run inference: image(s) + prompt → LaTeX string.
    images: PIL.Image or list[PIL.Image]
    """
    if not isinstance(images, list):
        images = [images]

    imgs_rgb = [img.convert("RGB") for img in images]
    inputs = processor(
        text=prompt_text,
        images=imgs_rgb,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    input_len = inputs["input_ids"].shape[1]
    new_tokens = generated_ids[:, input_len:]
    return processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()

### 2.2. Evaluation Metrics

We evaluate using four metrics: Exact Match (raw + normalized), CER, and BLEU. All metrics are defined in `metrics.py` and imported above. See `1_eda_and_setup.ipynb` (Section 2) for definitions, explanation, and sanity check. The choice of metrics — Exact Match (raw and normalized), Character Error Rate, and BLEU — follows the best-practice evaluation framework for Image-to-LaTeX systems (Orji et al., 2023).

> **Note:** `normalize_latex()` and `compute_metrics()` are imported from `metrics.py`. The sanity check is in `1_eda_and_setup.ipynb` (Section 2).

Metrics are working as expected.

### 2.3. One-Shot Example

The one-shot example (index 0 from the training set) was selected and visualized in `1_eda_and_setup.ipynb` (Section 3). Both model notebooks use the same index to ensure comparability. Here we reload it for use in inference. The one-shot prompting approach follows the in-context learning paradigm introduced by Brown et al. (2020).

For one-shot inference, we select a single example from the training set to include in the prompt as a demonstration of the expected input-output format. The choice of example matters: it should be representative of the dataset, visually clear, and not overly complex, so that the model can learn the output format without being distracted by content difficulty. We select a formula of moderate complexity from the beginning of the training set and visually verify its quality before use.

In [ ]:
# One-shot example (selected in 1_eda_and_setup.ipynb, Section 3)
ONE_SHOT_INDEX = 0
one_shot_example = dataset["train"][ONE_SHOT_INDEX]
one_shot_image = one_shot_example["image"].convert("RGB")
one_shot_text = one_shot_example["text"]
print(f"One-shot example: {one_shot_text}")

One-shot example: z _ { 1 } = r _ { 1 } ( \cos \theta _ { 1 } + i \sin \theta _ { 1 } )


The selected one-shot example is a medium-complexity formula containing subscripts, trigonometric functions, and parentheses, making it representative of the dataset.

### 2.4. Evaluation Pipeline: Running Inference and Computing Metrics

This section defines a reusable evaluation pipeline that runs inference on all 70 test examples and computes all four metrics. The pipeline accepts the model, processor, a prompt-building function, and the device as arguments, making it compatible with any model and any prompt strategy (zero-shot, one-shot, or fine-tuned). A progress bar is printed every 10 examples for monitoring.

In [ ]:
def run_evaluation(model, processor, test_data, prompt_fn, device="cpu",
                   max_new_tokens=256, verbose=True,
                   example_image=None, example_text=None):
    predictions = []
    references = [ex["text"] for ex in test_data]
    n = len(test_data)
    start_time = time.time()

    for i, ex in enumerate(test_data):
        try:
            # Zero-shot
            prompt_text = prompt_fn(processor)
            images = ex["image"]
        except TypeError:
            # One-shot
            prompt_text = prompt_fn(processor,
                                    example_image,
                                    example_text)
            images = [example_image, ex["image"]]

        pred = predict(images, prompt_text, model, processor, device, max_new_tokens)
        predictions.append(pred)

        if verbose and (i + 1) % 10 == 0:
            elapsed = time.time() - start_time
            print(f"  Processed {i+1}/{n} ({elapsed:.1f}s)")

    elapsed = time.time() - start_time
    if verbose:
        print(f"  Done. Total time: {elapsed:.1f}s ({elapsed/n:.1f}s per example)")

    return predictions, references, elapsed

### 2.5. SmolVLM-256M-Instruct (SmolVLM Team, 2025a)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the processor with explicit image size
# N=1 → longest_edge=512. This is sufficient for our images (~515×150).
processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    size={"longest_edge": 1 * 512}
)

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    attn_implementation="flash_attention_2" if device == "cuda" and torch.cuda.get_device_capability()[0] >= 8 else "eager",
).to(device)

model.eval()
print(f"Model: {MODEL_NAME}")
print(f"Device: {device}, dtype: {model.dtype}")

processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Some kwargs in processor config are unused and will not have any effect: image_seq_len. 


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/513M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Model: HuggingFaceTB/SmolVLM-256M-Instruct
Device: cuda, dtype: torch.bfloat16


#### 2.5.1. Zero-shot

In [ ]:
model_results = {}

print("Running zero-shot inference...")
zero_shot_preds, references, zs_time = run_evaluation(
    model, processor, dataset["test"],
    prompt_fn=build_zero_shot_prompt,
    device=device
)

Running zero-shot inference...
  Processed 10/70 (11.3s)
  Processed 20/70 (20.8s)
  Processed 30/70 (30.0s)
  Processed 40/70 (39.2s)
  Processed 50/70 (49.0s)
  Processed 60/70 (59.0s)
  Processed 70/70 (78.4s)
  Done. Total time: 78.4s (1.1s per example)


In [ ]:
zero_shot_metrics = compute_metrics(zero_shot_preds, references)
model_results["zero_shot"] = {
    "predictions": zero_shot_preds,
    "metrics": zero_shot_metrics
}

print(f"Zero-shot results:")
print(f"  Exact Match:          {zero_shot_metrics['exact_match']:.1%}")
print(f"  Exact Match (norm.):  {zero_shot_metrics['exact_match_normalized']:.1%}")
print(f"  CER:                  {zero_shot_metrics['cer']:.1%}")
print(f"  BLEU:                 {zero_shot_metrics['bleu']:.1f}")

print("\nExamples:")
for i in range(3):
    print(f"  REF: {references[i]}")
    print(f"  PRD: {zero_shot_preds[i]}")
    print()

Zero-shot results:
  Exact Match:          0.0%
  Exact Match (norm.):  20.0%
  CER:                  60.5%
  BLEU:                 49.7

Examples:
  REF: \sqrt { b ^ { 2 } - 4 a c }
  PRD: \sqrt{b^{2}-4ac}

  REF: \sqrt { x - y - z + x ^ { 2 } + y ^ { 2 } + z ^ { 2 } }
  PRD: \sqrt{x-y-z^{2}+x^{2}+y^{2}+z^{2}}

  REF: \frac { 2 \tan \alpha } { 1 - \tan ^ { 2 } \alpha }
  PRD: \frac{2tan\alpha}{1-tan^{\alpha}\alpha}



#### 2.5.2. One-shot

In [ ]:
print("Running one-shot inference...")
one_shot_preds, _, os_time = run_evaluation(
    model, processor, dataset["test"],
    prompt_fn=build_one_shot_prompt,
    device=device,
    example_image=one_shot_example["image"],
    example_text=one_shot_example["text"]
)

Running one-shot inference...
  Processed 10/70 (10.9s)
  Processed 20/70 (22.6s)
  Processed 30/70 (33.2s)
  Processed 40/70 (43.6s)
  Processed 50/70 (55.7s)
  Processed 60/70 (67.7s)
  Processed 70/70 (78.9s)
  Done. Total time: 78.9s (1.1s per example)


In [ ]:
one_shot_metrics = compute_metrics(one_shot_preds, references)
model_results["one_shot"] = {
    "predictions": one_shot_preds,
    "metrics": one_shot_metrics
}

print(f"One-shot results:")
print(f"  Exact Match:          {one_shot_metrics['exact_match']:.1%}")
print(f"  Exact Match (norm.):  {one_shot_metrics['exact_match_normalized']:.1%}")
print(f"  CER:                  {one_shot_metrics['cer']:.1%}")
print(f"  BLEU:                 {one_shot_metrics['bleu']:.1f}")

print("\nExamples:")
for i in range(3):
    print(f"  REF: {references[i]}")
    print(f"  PRD: {one_shot_preds[i]}")
    print()

One-shot results:
  Exact Match:          0.0%
  Exact Match (norm.):  15.7%
  CER:                  45.6%
  BLEU:                 42.4

Examples:
  REF: \sqrt { b ^ { 2 } - 4 a c }
  PRD: \sqrt{b^{2}-4ac}

  REF: \sqrt { x - y - z + x ^ { 2 } + y ^ { 2 } + z ^ { 2 } }
  PRD: \sqrt{x-y-z^{2}+x^{2}+y^{2}+z^{2}}

  REF: \frac { 2 \tan \alpha } { 1 - \tan ^ { 2 } \alpha }
  PRD: \frac{2tan\alpha}{1-tan^{\alpha}\alpha}



#### 2.5.3. Exporting Results

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)

save_data = {
    "model": MODEL_NAME,
    "device": device,
    "results": {
        "zero_shot": {
            "predictions": model_results["zero_shot"]["predictions"],
            "exact_match": model_results["zero_shot"]["metrics"]["exact_match"],
            "exact_match_normalized": model_results["zero_shot"]["metrics"]["exact_match_normalized"],
            "cer": model_results["zero_shot"]["metrics"]["cer"],
            "bleu": model_results["zero_shot"]["metrics"]["bleu"],
            "inference_time_per_example": round(zs_time / len(dataset["test"]), 1),
        },
        "one_shot": {
            "predictions": model_results["one_shot"]["predictions"],
            "exact_match": model_results["one_shot"]["metrics"]["exact_match"],
            "exact_match_normalized": model_results["one_shot"]["metrics"]["exact_match_normalized"],
            "cer": model_results["one_shot"]["metrics"]["cer"],
            "bleu": model_results["one_shot"]["metrics"]["bleu"],
            "inference_time_per_example": round(os_time / len(dataset["test"]), 1),
        },
        "references": references,
        "one_shot_example": {
            "index": ONE_SHOT_INDEX,
            "text": one_shot_example["text"],
        }
    }
}

save_path = os.path.join(SAVE_DIR, "smolvlm_zs_os_results.json")
with open(save_path, "w") as f:
    json.dump(save_data, f, ensure_ascii=False, indent=2)

print(f"Results saved to: {save_path}")

Results saved to: /content/drive/MyDrive/latex_ocr/smolvlm_zs_os_results.json


#### 2.5.4. Results

In [ ]:
results_table = pd.DataFrame([
    {
        "Model": MODEL_NAME.split("/")[-1],
        "Setup": "Zero-shot",
        "Exact Match": f"{model_results['zero_shot']['metrics']['exact_match']:.1%}",
        "EM (norm.)": f"{model_results['zero_shot']['metrics']['exact_match_normalized']:.1%}",
        "CER": f"{model_results['zero_shot']['metrics']['cer']:.1%}",
        "BLEU": f"{model_results['zero_shot']['metrics']['bleu']:.1f}",
    },
    {
        "Model": MODEL_NAME.split("/")[-1],
        "Setup": "One-shot",
        "Exact Match": f"{model_results['one_shot']['metrics']['exact_match']:.1%}",
        "EM (norm.)": f"{model_results['one_shot']['metrics']['exact_match_normalized']:.1%}",
        "CER": f"{model_results['one_shot']['metrics']['cer']:.1%}",
        "BLEU": f"{model_results['one_shot']['metrics']['bleu']:.1f}",
    }
])

print("Evaluation Results (before SFT)")
results_table

Evaluation Results (before SFT)


,Model,Setup,Exact Match,EM (norm.),CER,BLEU
0,SmolVLM-256M-Instruct,Zero-shot,0.0%,20.0%,60.5%,49.7
1,SmolVLM-256M-Instruct,One-shot,0.0%,15.7%,45.6%,42.4


Zero-shot and one-shot results reveal a nuanced picture. Neither setup achieves a raw exact match (EM = 0.0%), indicating that the 256M model cannot reproduce any formula character-for-character without fine-tuning. However, normalized EM reaches 20.0% for zero-shot and 15.7% for one-shot, showing that the model captures the correct mathematical structure to some extent, but still differs in formatting (whitespace, command variants).

The relationship between setups is mixed across metrics. Zero-shot outperforms one-shot on EM (normalized) and BLEU (49.7 vs 42.4), suggesting that the additional demonstration image in the one-shot prompt introduces noise for this small model — the two-image input likely confuses the limited attention capacity of a 256M-parameter architecture. Conversely, one-shot achieves a substantially lower CER (45.6% vs 60.5%), meaning it makes fewer character-level errors. A possible explanation is that the demonstration example helps the model adopt the correct tokenization style (e.g., spacing, command naming), reducing character insertions and deletions even when the overall sequence differs from the reference.

In summary, both baselines produce partially correct output but are far from usable. These results set a clear benchmark for SFT improvement.

## 3. Supervised Fine-Tuning

> **Dependencies.** This section relies on objects defined earlier in the notebook.
> To run Section 3 as a standalone block, ensure `dataset`, `mw_train_subsample`,
> `processor`, and all metric/helper functions from Section 2 are in scope.

### 3.1. SmolVLM-256M-Instruct using LoRA

This section performs SFT on SmolVLM-256M-Instruct using LoRA (Hu et al., 2022). The goal is to adapt the model to recognize handwritten mathematical formulas and output them in LaTeX format. We follow the official SmolVLM fine-tuning recipe: a custom data collator applies the chat template and constructs labels, while LoRA keeps the training memory-efficient. After training, we evaluate all epoch checkpoints on the validation set (68 examples) and select the best one for final evaluation on the test set (70 examples).

The same training pipeline is used twice with different training data:

- LaTeX_OCR only (1,164 examples)
- LaTeX_OCR + MathWriting-human (combined, 6,164 examples)

Switch between experiments by toggling `USE_COMBINED` in the configuration cell below. All code (LoRA config, data collation, training, checkpoint selection, evaluation) is shared.

In [ ]:
# ── Experiment selector ──────────────────────────────────────
# Change this flag to switch between training configurations.
#   False → LaTeX_OCR only (1,164 examples)
#   True  → LaTeX_OCR + MathWriting-human (combined, 6,164 examples)
USE_COMBINED = True
# ─────────────────────────────────────────────────────────────

In [ ]:
# Training
NUM_EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 8
LEARNING_RATE = 1e-4

In [ ]:
if USE_COMBINED:
    DATASET_NAME = "sft_2datasets"
    OUTPUT_DIR = "./smolvlm_lora_combined"
    ADAPTER_SAVE_NAME = "smolvlm_lora_2datasets"
    RESULTS_FILE = "smolvlm_lora_2datasets_results.json"

    train_dataset = data_prep.get_combined_train(dataset, mw_train_subsample)
    print(f"Training on COMBINED dataset: {len(train_dataset):,} examples")
else:
    DATASET_NAME = "sft_1dataset"
    OUTPUT_DIR = "./smolvlm_lora"
    ADAPTER_SAVE_NAME = "smolvlm_lora_1dataset"
    RESULTS_FILE = "smolvlm_lora_1dataset_results.json"

    train_dataset = dataset["train"]
    print(f"Training on LaTeX_OCR only: {len(train_dataset):,} examples")

train_dataset


Combined training set: 6,164 examples
Training on COMBINED dataset: 6,164 examples


Dataset({
    features: ['image', 'text'],
    num_rows: 6164
})

#### 3.1.1. LoRA Configuration and Model Loading

We use LoRA (Low-Rank Adaptation) to fine-tune only a small number of additional parameters while keeping the base model frozen. This drastically reduces VRAM usage and training time. The configuration follows the official SmolVLM fine-tuning recipe (SmolVLM Team, 2025b). Rank r=8 is a commonly used value recommended by Hu et al. (2022) and has been shown to achieve a good balance between adaptation capacity and overfitting risk for parameter-efficient fine-tuning. Targeting all linear projection layers (q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj) is the standard approach for vision-language models (Hu et al., 2022; Mangrulkar et al., 2022). The Instruct variant is used as the starting checkpoint because it already understands chat-based instructions.

In [ ]:
# Free the model from Step 2 to release GPU memory
if "model" in dir():
    del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    size={"longest_edge": 512}
)

Some kwargs in processor config are unused and will not have any effect: image_seq_len. 


In [ ]:
# LoRA config following the official SmolVLM fine-tuning recipe
lora_config = LoraConfig(
    r=8,
    lora_alpha=8,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    init_lora_weights="gaussian",
)

In [ ]:
# Load a fresh model and wrap it with LoRA adapters
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2" if device == "cuda" and torch.cuda.get_device_capability()[0] >= 8 else "eager",
).to(device)

model = get_peft_model(model, lora_config)
model.enable_input_require_grads()  # Required for LoRA + gradient checkpointing
model.print_trainable_parameters()

trainable params: 2,884,608 || all params: 259,369,536 || trainable%: 1.1122


#### 3.1.2. Data Collation

A custom collator function is responsible for formatting each batch. For every training example, it constructs a two-turn conversation (user instruction with the image → assistant response with the ground-truth LaTeX) and applies the model's chat template. The processor then tokenizes the text and processes the images in one call. Labels are derived from input_ids by masking two token types with -100 (which excludes them from the loss): padding tokens and <image> tokens. This ensures the model learns to predict only the text tokens — both the instruction and the LaTeX response.



In [ ]:
# Token ID for <image> — will be masked in labels so the model
# does not try to predict image placeholder tokens.
image_token_id = processor.tokenizer.additional_special_tokens_ids[
    processor.tokenizer.additional_special_tokens.index("<image>")
]

def collate_fn(examples):
    """Formats a batch of (image, text) pairs into model inputs with labels."""
    texts = []
    images = []

    for example in examples:
        image = example["image"]
        if image.mode != "RGB":
            image = image.convert("RGB")

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": (
                        "Convert this handwritten mathematical formula "
                        "to LaTeX format. Output only the LaTeX code, nothing else."
                    )},
                ]
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": example["text"]}]
            }
        ]

        # add_generation_prompt=False → include the assistant response in the text
        text = processor.apply_chat_template(messages, add_generation_prompt=False)
        texts.append(text.strip())
        images.append([image])  # list of images per sample (single image here)

    # Processor tokenizes text and encodes images in one call
    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)

    # Build labels: copy input_ids, then mask padding and image tokens
    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100   # ignore padding
    labels[labels == image_token_id] = -100                     # ignore image tokens
    batch["labels"] = labels

    return batch

#### 3.1.3. Training

Training hyperparameters (epochs, batch size, learning rate, etc.) are defined in the configuration cell at the top of this section. A checkpoint is saved after each epoch (save_strategy="epoch") so that we can later select the best one on the validation set, effectively implementing a post-hoc early stopping strategy (Prechelt, 1998). The learning rate of 1e-4 is the default recommended for LoRA fine-tuning in the original LoRA paper (Hu et al., 2022) and has been adopted by most VLM fine-tuning recipes, including the SmolVLM fine-tuning guide. Three epochs were chosen based on the observation that small datasets (< 10K examples) tend to overfit quickly — Orji et al. (2023) report diminishing returns after 3–5 epochs for Image-to-LaTeX tasks. Effective batch size 16 is a practical choice dictated by GPU memory constraints (T4 GPU with 16 GB VRAM) and aligns with common practice in VLM fine-tuning literature. remove_unused_columns=False is required because the dataset contains image and text columns that are not direct model inputs but are needed by the custom collator.

In [ ]:
training_args = TrainingArguments(
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_ratio=0.1,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    logging_steps=25,
    save_strategy="epoch",
    save_total_limit=3,
    bf16=True,
    output_dir=OUTPUT_DIR,
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    train_dataset=train_dataset,
)

trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss
25,35.256600
50,23.373200
75,9.565000
100,4.180000
125,3.100300
150,2.420500
175,1.798900
200,0.862700
225,0.731600
250,0.648300


Step,Training Loss
25,35.256600
50,23.373200
75,9.565000
100,4.180000
125,3.100300
150,2.420500
175,1.798900
200,0.862700
225,0.731600
250,0.648300


TrainOutput(global_step=1155, training_loss=2.0769127785901484, metrics={'train_runtime': 11230.2808, 'train_samples_per_second': 1.647, 'train_steps_per_second': 0.103, 'total_flos': 2989883001415680.0, 'train_loss': 2.0769127785901484, 'epoch': 2.99805321219987})

#### 3.1.4. Checkpoint Selection on Validation Set (Prechelt, 1998)

Training longer does not always produce the best model, a phenomenon well-documented in the early stopping literature (Prechelt, 1998). We therefore evaluate all three epoch checkpoints on the 68-example validation set and select the best one by **normalized EM** (primary) and **CER** (tiebreaker — lower is better). Normalized EM is the standard equation-level accuracy metric for Image-to-LaTeX systems (Orji et al., 2023, §12), while CER provides a symbol-level distance measure that is more appropriate as a tiebreaker than BLEU, since CER directly penalizes structural errors (e.g., missing braces) that BLEU may overlook. The best checkpoint is then used for the final test evaluation.

In [ ]:
# Free the trained model to release GPU memory before evaluation
del model
gc.collect()
torch.cuda.empty_cache()

checkpoint_dirs = sorted(glob(f"{OUTPUT_DIR}/checkpoint-*"))
print(f"Found {len(checkpoint_dirs)} checkpoints")

# Evaluate each checkpoint on the validation set
val_results = {}

for ckpt_dir in checkpoint_dirs:
    print(f"\nEvaluating {ckpt_dir} ...")

    base_model = AutoModelForVision2Seq.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2" if device == "cuda" and torch.cuda.get_device_capability()[0] >= 8 else "eager",
    ).to(device)

    model_ckpt = PeftModel.from_pretrained(base_model, ckpt_dir)
    model_ckpt.eval()

    preds, refs, _ = run_evaluation(
        model_ckpt, processor, dataset["validation"],
        prompt_fn=build_zero_shot_prompt,
        device=device, verbose=False,
    )

    metrics = compute_metrics(preds, refs)
    epoch = ckpt_dir.split("-")[-1]
    val_results[epoch] = {
        "checkpoint": ckpt_dir,
        **metrics,
    }
    print(f"  EM: {metrics['exact_match']:.1%}, "
          f"EM (norm.): {metrics['exact_match_normalized']:.1%}, "
          f"CER: {metrics['cer']:.1%}, "
          f"BLEU: {metrics['bleu']:.1f}")

    # Free memory for next checkpoint
    del base_model, model_ckpt
    gc.collect()
    torch.cuda.empty_cache()

# Select best checkpoint by normalized EM, then CER as tiebreaker
best_epoch = max(val_results, key=lambda e: (val_results[e]["exact_match_normalized"],
                                              -val_results[e]["cer"]))
best_ckpt = val_results[best_epoch]["checkpoint"]
print(f"\nBest checkpoint: epoch {best_epoch} ({best_ckpt})")
print(f"  EM: {val_results[best_epoch]['exact_match']:.1%}, "
      f"EM (norm.): {val_results[best_epoch]['exact_match_normalized']:.1%}, "
      f"CER: {val_results[best_epoch]['cer']:.1%}, "
      f"BLEU: {val_results[best_epoch]['bleu']:.1f}")

Found 3 checkpoints

Evaluating ./smolvlm_lora_combined/checkpoint-1155 ...
  EM: 75.0%, EM (norm.): 75.0%, CER: 6.1%, BLEU: 92.3

Evaluating ./smolvlm_lora_combined/checkpoint-385 ...
  EM: 48.5%, EM (norm.): 48.5%, CER: 17.4%, BLEU: 80.9

Evaluating ./smolvlm_lora_combined/checkpoint-770 ...
  EM: 72.1%, EM (norm.): 72.1%, CER: 7.0%, BLEU: 91.0

Best checkpoint: epoch 1155 (./smolvlm_lora_combined/checkpoint-1155)
  EM: 75.0%, EM (norm.): 75.0%, CER: 6.1%, BLEU: 92.3


#### 3.1.5. Evaluation on Test Set (70 Examples)

The selected checkpoint is evaluated on the same 70 test examples from linxy/LaTeX_OCR using the same metrics as in Step 2 (EM, EM normalized, CER, BLEU). This ensures fair comparison across all four setups: zero-shot, one-shot, SFT (1 dataset) and SFT (2 datasets).

In [ ]:
print(f"Loading best checkpoint: {best_ckpt} ...")

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2" if device == "cuda" and torch.cuda.get_device_capability()[0] >= 8 else "eager",
).to(device)

model_sft = PeftModel.from_pretrained(base_model, best_ckpt)
model_sft.eval()

print("Running SFT evaluation on test set (70 examples)...")
sft_preds, sft_refs, sft_time = run_evaluation(
    model_sft, processor, dataset["test"],
    prompt_fn=build_zero_shot_prompt,
    device=device,
)

In [ ]:
sft_metrics = compute_metrics(sft_preds, sft_refs)

print(f"\n{DATASET_NAME} results:")
print(f"  Exact Match:          {sft_metrics['exact_match']:.1%}")
print(f"  Exact Match (norm.):  {sft_metrics['exact_match_normalized']:.1%}")
print(f"  CER:                  {sft_metrics['cer']:.1%}")
print(f"  BLEU:                 {sft_metrics['bleu']:.1f}")

print("\nExamples:")
for i in range(5):
    print(f"  REF: {sft_refs[i]}")
    print(f"  PRD: {sft_preds[i]}")
    print()


sft_2datasets results:
  Exact Match:          80.0%
  Exact Match (norm.):  80.0%
  CER:                  4.1%
  BLEU:                 95.0

Examples:
  REF: \sqrt { b ^ { 2 } - 4 a c }
  PRD: \sqrt { b ^ { 2 } - 4 a c }

  REF: \sqrt { x - y - z + x ^ { 2 } + y ^ { 2 } + z ^ { 2 } }
  PRD: \sqrt { x - y - z + x ^ { 2 } + y ^ { 2 } + z ^ { 2 } }

  REF: \frac { 2 \tan \alpha } { 1 - \tan ^ { 2 } \alpha }
  PRD: \frac { 2 \tan \alpha } { 1 - \tan ^ { 2 } \alpha }

  REF: \lim _ { x \rightarrow - 1 } \frac { x ^ { 3 } + 1 } { x + 1 }
  PRD: \lim _ { x \rightarrow - 1 } \frac { x ^ { 3 } + 1 } { x + 1 }

  REF: 1 + \frac { 1 } { 1 ! } + \frac { 1 } { 2 ! } + \frac { 1 } { 3 ! } + \frac { 1 } { 4 ! }
  PRD: 1 + \frac { 1 } { 1 } + \frac { 1 } { 2 } + \frac { 1 } { 3 } + \frac { 1 } { 4 ! }



#### 3.1.6. Exporting Results

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)

# Save LoRA adapter weights to Google Drive
adapter_save_path = os.path.join(SAVE_DIR, ADAPTER_SAVE_NAME)
model_sft.save_pretrained(adapter_save_path)
processor.save_pretrained(adapter_save_path)
print(f"LoRA adapter saved to: {adapter_save_path}")

# Save numerical results
results_this_step = {
    "predictions": sft_preds,
    "exact_match": sft_metrics["exact_match"],
    "exact_match_normalized": sft_metrics["exact_match_normalized"],
    "cer": sft_metrics["cer"],
    "bleu": sft_metrics["bleu"],
    "best_epoch": int(best_epoch),
    "validation_results": val_results,
    "inference_time_per_example": round(sft_time / len(dataset["test"]), 1),
}

save_path = os.path.join(SAVE_DIR, RESULTS_FILE)
with open(save_path, "w") as f:
    json.dump({"experiment": DATASET_NAME, "results": results_this_step}, f, ensure_ascii=False, indent=2)
print(f"Results saved to: {save_path}")

LoRA adapter saved to: /content/drive/MyDrive/latex_ocr/smolvlm_lora_2datasets
Results saved to: /content/drive/MyDrive/latex_ocr/smolvlm_lora_2datasets_results.json


## 4. Results Summary

This section aggregates all results from the SmolVLM-256M-Instruct experimental pipeline — zero-shot, one-shot, and both SFT configurations — into a single comparison table. Results are loaded from Google Drive, so the notebook can be run incrementally: first for zero-shot/one-shot and SFT on a single dataset, then again for SFT on the combined dataset.

In [ ]:
rows = []

# Zero-shot & One-shot
zs_os_path = os.path.join(SAVE_DIR, "smolvlm_zs_os_results.json")
if os.path.exists(zs_os_path):
    with open(zs_os_path) as f:
        zs_os = json.load(f)
    for key, label in [("zero_shot", "Zero-shot"), ("one_shot", "One-shot")]:
        r = zs_os["results"][key]
        rows.append({"Setup": label,
                      "Exact Match": r['exact_match'],
                      "EM (norm.)": r['exact_match_normalized'],
                      "CER": r['cer'],
                      "BLEU": r['bleu'],
                      "Time (s/ex)": r['inference_time_per_example']})
    print(f"Loaded ZS/OS results from: {zs_os_path}")
else:
    print(f"ZS/OS results not found: {zs_os_path}")
    for label in ["Zero-shot", "One-shot"]:
        rows.append({"Setup": label, "Exact Match": None, "EM (norm.)": None,
                      "CER": None, "BLEU": None, "Time (s/ex)": None})

# SFT results
for fname, label in [("smolvlm_lora_1dataset_results.json", "SFT (LaTeX_OCR)"),
                      ("smolvlm_lora_2datasets_results.json", "SFT (combined)")]:
    path = os.path.join(SAVE_DIR, fname)
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        r = data["results"]
        rows.append({"Setup": label,
                      "Exact Match": r['exact_match'],
                      "EM (norm.)": r['exact_match_normalized'],
                      "CER": r['cer'],
                      "BLEU": r['bleu'],
                      "Time (s/ex)": r.get("inference_time_per_example")})
        print(f"Loaded SFT results from: {path}")
    else:
        rows.append({"Setup": label, "Exact Match": None, "EM (norm.)": None,
                      "CER": None, "BLEU": None, "Time (s/ex)": None})
        print(f"SFT results not found yet: {path}")

results_summary = pd.DataFrame(rows)

metric_cols = ["Exact Match", "EM (norm.)", "CER", "BLEU", "Time (s/ex)"]

(results_summary.style
    .background_gradient(subset=metric_cols, cmap="coolwarm", axis=0)
    .format({"Exact Match": "{:.1%}", "EM (norm.)": "{:.1%}", "CER": "{:.1%}",
             "BLEU": "{:.1f}", "Time (s/ex)": "{:.1f}"})
    .hide(axis="index")
)

Loaded ZS/OS results from: /content/drive/MyDrive/latex_ocr/smolvlm_zs_os_results.json
Loaded SFT results from: /content/drive/MyDrive/latex_ocr/smolvlm_lora_1dataset_results.json
Loaded SFT results from: /content/drive/MyDrive/latex_ocr/smolvlm_lora_2datasets_results.json


Setup,Exact Match,EM (norm.),CER,BLEU,Time (s/ex)
Zero-shot,0.0%,20.0%,60.5%,49.7,1.1
One-shot,0.0%,15.7%,45.6%,42.4,1.1
SFT (LaTeX_OCR),70.0%,70.0%,8.1%,91.4,2.6
SFT (combined),80.0%,80.0%,4.1%,95.0,2.4


LoRA fine-tuning transforms the model from unusable to near-usable. Raw exact match jumps from 0.0% (both baselines) to 70.0% on LaTeX_OCR alone and 80.0% on the combined dataset (LaTeX_OCR + MathWriting, 6,164 examples total). CER drops from 45.6–60.5% to 8.1% and 4.1% respectively. EM and EM (norm.) are identical for both SFT rows, meaning every correct prediction is character-perfect — unlike the baselines, where normalisation uncovered partially correct structure hidden behind formatting differences.

The combined dataset consistently outperforms single-dataset training: +10 pp EM, −4.0 pp CER, +3.6 BLEU. Both configurations used the same hyperparameters (3 epochs, batch 2 × grad accum 8), so the improvement comes from data diversity rather than longer training. Exposure to MathWriting's different notation conventions appears to act as regularisation, helping the LoRA adapters generalise better.

Inference time on a T4 GPU increases from 1.1 s/example (baselines) to 2.4–2.6 s/example after SFT. The difference between the two SFT configurations (2.4 vs. 2.6) is within measurement noise. Both SFT configurations maintain practical inference speed (2.4–2.6 s/example on a T4 GPU), suitable for batch processing of scanned documents.

## 5. Conclusion

SmolVLM-256M-Instruct with LoRA fine-tuning achieves 70% EM (CER 8.1%) on LaTeX_OCR alone and 80% EM (CER 4.1%) when trained on the combined 6,164-example dataset. The cross-dataset gain (+10 pp EM) confirms that even a small auxiliary dataset with different notation conventions improves generalisation. Without fine-tuning, the model is unable to reproduce any formula exactly (EM = 0%, CER ≈ 46–61%).

These results establish SmolVLM-256M + LoRA as a low-resource baseline. In the next step, we evaluate whether Qwen3-VL-2B-Instruct can push these metrics higher.

## References

1. Orji, E.Z., Haydar, A., Erşan, İ., Mwambe, O.O. (2023). Advancing OCR Accuracy in Image-to-LaTeX Conversion—A Critical and Creative Exploration. *Applied Sciences*, 13(22), 12503. https://doi.org/10.3390/app132212503
2. Hu, E.J., Shen, Y., Wallis, P., Allen-Zhu, Z., Li, Y., Wang, S., Wang, L., Chen, W. (2022). LoRA: Low-Rank Adaptation of Large Language Models. *ICLR 2022*. https://arxiv.org/abs/2106.09685
3. SmolVLM Team @ HuggingFace. (2025a). SmolVLM: A Small Yet Capable Vision Language Model. HuggingFace Model Card. https://huggingface.co/HuggingFaceTB/SmolVLM-256M-Instruct
4. SmolVLM Team @ HuggingFace. (2025b). Fine-Tuning SmolVLM on DocVQA. HuggingFace Blog. https://huggingface.co/blog/smolvlm
5. Brown, T.B., Mann, B., Ryder, N., et al. (2020). Language Models are Few-Shot Learners. *NeurIPS 2020*, 33, 1877–1901. https://arxiv.org/abs/2005.14165
6. Prechelt, L. (1998). Early Stopping — But When? In *Neural Networks: Tricks of the Trade*. Springer, LNCS 1524, pp. 55–69. https://doi.org/10.1007/3-540-49430-8_3
7. Mangrulkar, S., Gugger, S., Debut, L., Belkada, Y., Paul, S. (2022). PEFT: State-of-the-art Parameter-Efficient Fine-Tuning Methods. GitHub / HuggingFace. https://github.com/huggingface/peft
8. linxy. (2024). linxy/LaTeX_OCR (Human Handwrite subset). HuggingFace Datasets. https://huggingface.co/datasets/linxy/LaTeX_OCR
9. deepcopy. (2024). deepcopy/MathWriting-human. HuggingFace Datasets. https://huggingface.co/datasets/deepcopy/MathWriting-human